# Returns, volatility and drawdowns

In [ ]:
from pathlib import Path
import sys

# Works whether Jupyter starts in the project root or in notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "config.json").exists() and (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.data import read_config, download_data
config = read_config(ROOT)


In [ ]:
from src.indicators import add_features
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

data = download_data(ROOT, config)
features = {ticker: add_features(bars, config) for ticker, bars in data.items()}
features["MU"][["daily_return", "sma_fast", "sma_slow", "momentum", "rsi", "rolling_volatility"]].tail().round(4)

In [ ]:
start = config["common_window"]["start"]
end = config["common_window"]["end_exclusive"]
prices = pd.concat({ticker: frame["adj_close"] for ticker, frame in data.items()}, axis=1)
common = prices.loc[(prices.index >= start) & (prices.index < end)]
assert common.notna().all().all()
# This descriptive EDA line is rebased at the first CLOSE, without costs.
normalized = common / common.iloc[0]
fig, ax = plt.subplots(figsize=(10, 4))
normalized.plot(ax=ax)
ax.set(title="Common-date adjusted prices, rebased to 1 at the first close",
       ylabel="Price ratio (descriptive, before costs)")
ax.grid(alpha=.2)
plt.show()

In [ ]:
drawdowns = normalized / normalized.cummax() - 1
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
drawdowns.plot(ax=axes[0])
axes[0].set(title="Descriptive price drawdown", ylabel="Drawdown")
for ticker in config["tickers"]:
    vol = features[ticker]["rolling_volatility"].reindex(common.index)
    axes[1].plot(vol.index, vol, label=ticker)
axes[1].set(title="20-session rolling volatility", ylabel="Annualized volatility")
for ax in axes:
    ax.yaxis.set_major_formatter(PercentFormatter(1)); ax.grid(alpha=.2)
axes[1].legend()
fig.tight_layout()
plt.show()

In [ ]:
# Keep the first return unavailable: it precedes this rebased EDA interval.
returns = common.pct_change(fill_method=None)
returns.corr().round(3)